# archscope quickstart

Cross-architecture mechanistic interpretability on small models.

**In this notebook you will:**

1. Install archscope (PyPI, ~5s)
2. Load Pythia-160m (transformer) and extract its residual stream
3. Train a sentiment probe on one layer
4. Load Mamba-130m and extract its recurrent SSM state `h_T`
5. Transfer the probe direction from Mamba into Pythia activation space

Runs end-to-end on Colab's free CPU tier in ~5 minutes.

- GitHub: https://github.com/OriginalKazdov/archscope
- Docs: https://github.com/OriginalKazdov/archscope/tree/main/docs
- Full reproducible cross-arch experiment: `examples/cross_arch_sentiment_transfer.py`

## 1. Install

In [ ]:
!pip install -q archscope transformers

In [ ]:
import torch
import archscope as ai

print("archscope:", ai.__version__)

## 2. Load a transformer model

`ai.load_model` returns `(model, tokenizer, backend)` in one call. It sets `pad_token = eos_token` if missing, calls `model.eval()`, and picks the right backend for the architecture.

In [ ]:
model, tok, backend = ai.load_model("EleutherAI/pythia-160m", arch="transformer")

print("backend:", type(backend).__name__)
print("layers available:", len(backend.layer_names()))
print("first 3:", backend.layer_names()[:3])

## 3. Extract activations

`backend.extract(inputs, layers=[...])` runs a single forward pass and returns one `ActivationRecord` per requested layer.

In [ ]:
inputs = tok(["The capital of France is", "Music is the food of"],
             return_tensors="pt", padding=True)

records = backend.extract(inputs, layers=["layer_5.residual"])
print("shape (B, T, H):", tuple(records[0].activations.shape))
print("meta:", records[0].meta)

## 4. Train a sentiment probe

`fit_probe` handles tokenization + extraction + training. Use the `pos_texts` / `neg_texts` convention for a quick test. The probe is linear by default.

In [ ]:
pos = ["I love this movie", "Amazing show", "Wonderful day",
       "Fantastic work", "Truly delightful"]
neg = ["I hate this movie", "Awful film", "Terrible day",
       "Disappointing work", "Truly dreadful"]

pf = ai.probes.fit_probe(
    model,
    tokenizer=tok,
    pos_texts=pos, neg_texts=neg,
    layer_name="layer_7.residual",
    backend_hint="transformer",
)

print("metrics:", pf.metrics)
print("direction shape:", tuple(pf.direction.shape))
print("bias:", pf.bias.item())

10 examples is tiny — `val_auroc=0.5` is the documented edge case (only one class in the val split). The point of this cell is the API, not the metric.

`pf.direction` is the 1D weight vector in activation space; `pf.bias` is the scalar offset. For any activation `a`, the probe logit is exactly `a @ pf.direction + pf.bias`. This decoupling matters for the cross-arch step at the end of the notebook.

## 5. Logit lens — what does each layer 'think'?

Project each layer's residual stream through the model's own final norm + unembedding.

In [ ]:
result = ai.lens.logit_lens(
    model, tok,
    prompt="The capital of France is",
    target_token=" Paris",
    backend_hint="transformer",
)
print(result.to_markdown())

Read the trajectory: target rank should drop substantially as you go deeper into the network. Pythia-160m is too small to surface ` Paris` as top-1, but the rank moves from ~5000 down to ~80.

**Caveat for Mamba**: naive logit lens degrades with depth on SSMs because the residual stream isn't trained to be unembed-decodable at every layer. Use `ai.lens.TunedLens.fit(...)` for deep-layer readouts on Mamba.

## 6. Switch to Mamba — extract the recurrent SSM state

Mamba exposes two kinds of state per block: `.residual` (the residual stream, shape `(B, T, H)`) and `.ssm_state` (the **final recurrent state `h_T`** after processing the whole sequence — Mamba's analog of an RNN's last hidden state).

In [ ]:
m_model, m_tok, m_backend = ai.load_model("state-spaces/mamba-130m-hf", arch="mamba")

print("first 4 layer names:")
for ln in m_backend.layer_names()[:4]:
    print(" ", ln)

In [ ]:
m_inputs = m_tok(["The cat sat on the mat"], return_tensors="pt", padding=True)

rec = m_backend.extract(m_inputs, layers=["layer_12.ssm_state"])[0]
print("ssm_state shape (B, d_inner, d_state):", tuple(rec.activations.shape))
print("meta:", rec.meta["shape_meaning"])

Shape is `(1, 1536, 16)` for mamba-130m — `1536 = d_inner`, `16 = d_state`. There is no sequence axis: this is *one* recurrent state per example, summarising everything the block has read.

Other libraries make this extraction awkward — it requires manually wiring a `DynamicCache` and threading `use_cache=True` through Mamba's HF wrapper. archscope hides that behind `backend.extract`.

## 7. Cross-architecture transfer — the main wedge

Train a probe direction on one architecture, learn a linear alignment to map activations from another, and check whether the direction still classifies. The whole point of having a unified backend API is that this fits in ~20 lines.

In [ ]:
from archscope.transfer import learn_alignment

# Paired activations on the same texts — that's what 'paired' means.
align_texts = ["The cat sat on the mat.", "Music has power.",
               "Solve for x.", "Birds sing at dawn.",
               "I love this movie", "I hate this movie",
               "Wonderful day", "Terrible day"]

def pool(backend, tokr, texts, layer):
    inp = tokr(texts, return_tensors="pt", padding=True)
    rec = backend.extract(inp, layers=[layer])[0]
    return rec.activations.mean(dim=1).detach()   # mean over seq

src_paired = pool(backend,   tok,   align_texts, "layer_7.residual")
tgt_paired = pool(m_backend, m_tok, align_texts, "layer_23.residual")

M = learn_alignment(src_paired, tgt_paired, ridge=1e-3)
print("alignment matrix shape (d_src, d_tgt):", tuple(M.shape))

`M` has shape `(d_src, d_tgt)` and satisfies `src_act ≈ M @ tgt_act`. To carry the **Pythia probe** into **Mamba space**, you transport the direction:

In [ ]:
# Transport probe: w_mamba = M.T @ w_pythia
w_mamba = M.T @ pf.direction
b_mamba = pf.bias        # bias is preserved
print("transported direction shape:", tuple(w_mamba.shape))

# Score a fresh Mamba activation directly
test_texts = ["Wonderful experience", "Terrible experience"]
test_tgt = pool(m_backend, m_tok, test_texts, "layer_23.residual")
logits = test_tgt @ w_mamba + b_mamba
probs = torch.sigmoid(logits)

for text, p in zip(test_texts, probs):
    print(f"  '{text}' -> p(positive) = {p.item():.3f}")

The probe was never trained on a single Mamba activation. Whatever signal you see here is purely the geometric overlap between Pythia layer 7 and Mamba layer 23 — a linear map between architectures.

For a proper study with 80/20 splits, 3 seeds, and quantitative AUROC numbers, see `examples/cross_arch_sentiment_transfer.py` in the repo.

## Where to go next

- **Cookbook** (recipes): https://github.com/OriginalKazdov/archscope/blob/main/docs/cookbook.md
- **Tutorial** (walkthrough): https://github.com/OriginalKazdov/archscope/blob/main/docs/tutorial.md
- **API reference**: https://github.com/OriginalKazdov/archscope/blob/main/docs/api.md
- **Full cross-arch experiment script**: https://github.com/OriginalKazdov/archscope/blob/main/examples/cross_arch_sentiment_transfer.py

Things archscope is built for: cross-arch experiments on small (≤1B) models, Mamba SSM-state extraction, probe transfer across architectures, behavioural circuit detection across families.

Things to use other libraries for: transformer-only work on large models (try `transformer_lens` or `nnsight`), production model auditing, SAE training at scale.